# Principal Component Analysis

The intention of this notebook is to perform the PCA analysis on genotype data and generate plots.

## Description

This notebook performs PCA on genotype data of **unrelated** individuals and projects the remaining (related) samples back into that PCA space. The main steps are: remove related individuals, LD-prune the variants, run PCA on the unrelated set, and exclude PCA-space outliers.

Relatedness estimation and sample QC are done upstream in `GWAS_QC.ipynb`.

## Input Files

| File | Description |
|------|-------------|
| `output/gwas_qc/genotype/protocol_example.merged.plink_qc.protocol_example.king.unrelated.plink_qc.prune.bed` | LD-pruned PLINK bundle for **unrelated** individuals (from GWAS_QC). PCA is computed on these samples. |
| `output/gwas_qc/kinship/protocol_example.merged.plink_qc.protocol_example.king.related.bed` | PLINK bundle for **related** individuals (from GWAS_QC), projected back onto the PCA model. |
| `input/covariate/protocol_example.pca_pheno.txt` | Phenotype/label table with an `IID` column (plus optional `FID`) used to label/colour the projected samples. A PLINK `.fam` file may be used instead. |

Genotypes must be in PLINK format (common variants, LD pruned) and split into related and unrelated sets. The phenotype file may also carry population/disease columns for labelling the PCA plots.

## Output Files

| File | Description |
|------|-------------|
| `output/pca_uf/*.pca.rds` | PCA model + PC scores for the unrelated samples (RDS). |
| `output/pca_uf/*.pca.txt` | PC scores table for the unrelated samples. |
| `output/pca_uf/*.pca.scree.png` | Scree plot of variance explained. |
| `output/pca_uf/*.pca.mahalanobis.*`, `*.pca.outliers` | Mahalanobis distances per population and the list of PC outliers. |
| `output/pca_uf/*.projected.rds` / `*.projected.txt` | Projected PC scores for the related individuals. |
| `output/pca_uf/*.projected.pc.png` | PCA plot including the projected related samples. |

## Minimal Working Example

The data can be found on [Synapse](https://www.synapse.org/#!Synapse:syn69670658/files/).

**Note: parameters set for the MWE are meant to let the MWE work to show the workflow procedures. They may be unrealistic and should not be used in practice. The pipeline has reasonable default values for what we suggest to use in practice for most of the parameters.**

### **Step 1.** Estimate kinship in the sample (prerequisite)

Before PCA we need to know which individuals are related, because PCA must be computed on an **unrelated** subset and the related individuals are projected back afterwards. This step runs the `king` workflow from GWAS_QC to estimate pairwise kinship and split the samples into *unrelated* and *related* sets. It is an upstream prerequisite — its outputs (the `king.unrelated` and `king.related` PLINK bundles) feed the PCA steps below.

In [ ]:
sos run pipeline/GWAS_QC.ipynb king \
    --cwd output/gwas_qc/kinship \
    --genoFile output/gwas_qc/plink/protocol_example.merged.plink_qc.bed \
    --name protocol_example.king \
    --keep-samples output/gwas_qc/genotype/protocol_example.rnaseq.bed.sample_genotypes.txt

### **Step 2.** Sample selection and QC of the genotype data (prerequisite)

QC the genotypes that will go into PCA: filter on MAF, sample/variant missingness and Hardy-Weinberg equilibrium, and LD-prune the variants. You can restrict to one population or to the related / unrelated split. Here we QC and LD-prune the **unrelated** individuals to build the PCA reference, and separately extract the same variants for the related individuals so they can be projected later. These `qc` / `qc_no_prune` calls come from GWAS_QC and are upstream prerequisites for the PCA steps.


#### Variant level and sample level QC on unrelated individuals, in preparation for PCA analysis

In [ ]:
sos run pipeline/GWAS_QC.ipynb qc \
    --cwd output/gwas_qc/genotype \
    --genoFile output/gwas_qc/kinship/protocol_example.merged.plink_qc.protocol_example.king.unrelated.bed \
    --mac-filter 5

Extract the previously selected (LD-pruned) variants from the **related** individuals, applying only a sample-level missingness filter so the related samples carry exactly the same variant set as the unrelated reference.

In [ ]:
sos run pipeline/GWAS_QC.ipynb qc_no_prune \
    --cwd output/gwas_qc/cache \
    --genoFile output/genotype_formatting/plink/protocol_example.merged.bed \
    --geno-filter 0 --mind-filter 0.1 --maf-filter 0 \
    --keep-variants output/gwas_qc/genotype/protocol_example.merged.plink_qc.protocol_example.king.unrelated.plink_qc.prune.in \
    --name for_pca

### **Step 3.** Run PCA on the unrelated samples (flashpca)

This is the core PCA step. It runs `flashpca` on the LD-pruned **unrelated** PLINK bundle to compute the principal components, then automatically computes Mahalanobis distances and flags PC outliers. Outputs: the fitted PCA model (`.pca.rds`), the PC score table (`.pca.txt`), a scree plot of variance explained, and the outlier list. Timing: <2 min.

The `--name` argument tags all output files; point `--genoFile` at the LD-pruned unrelated bed produced upstream.

In [ ]:
sos run pipeline/PCA.ipynb flashpca \
    --cwd output/pca_uf \
    --genoFile output/gwas_qc/genotype/protocol_example.merged.plink_qc.protocol_example.king.unrelated.plink_qc.prune.bed \
    --name protocol_example

**What you should see:** the workflow runs `flashpca_1` (PCA), then `detect_outliers` (Mahalanobis distance per population) and `plot_pca`, finishing with "executed successfully with 5 completed steps". The PCA model and PC scores are written to the `--cwd` directory. The scree plot below shows the proportion of variance explained by each PC — use it to decide how many PCs to keep.

In [ ]:
%preview output/pca_uf/protocol_example.merged.plink_qc.protocol_example.king.unrelated.plink_qc.prune.protocol_example.pca.scree.png

### **Step 4.** Project the related individuals onto the PCA model

The related individuals were held out of the PCA, so we now project them onto the model fitted from the unrelated samples. First reduce the related genotypes to **exactly** the same pruned variant set used to build the model (the `plink2 --extract` cell below) — otherwise the projection errors with "Input number of variants should be the same as used in the previous PCA model". Then run `project_samples`, passing a phenotype table (must contain an `IID` column) and, if available, a population/ethnicity column via `--label-col` / `--pop-col` (here the `race` column) so the projected samples can be coloured and the outlier detection can run.

In [ ]:
sos run pipeline/PCA.ipynb project_samples \
    --cwd output/pca_uf \
    --genoFile output/gwas_qc/genotype/protocol_example.merged.plink_qc.protocol_example.king.unrelated.plink_qc.prune.bed \
    --phenoFile input/covariate/protocol_example.pca_pheno.txt \
    --pca-model output/pca_uf/protocol_example.merged.plink_qc.protocol_example.king.unrelated.plink_qc.prune.protocol_example.pca.rds \
    --label-col race \
    --pop-col race \
    --maha-k 2

In [ ]:
%preview output/pca_uf/protocol_example.pca_pheno.pca.projected.pc.png

In [ ]:
sos run pipeline/GWAS_QC.ipynb qc_no_prune \
    --cwd output/pca_related \
    --genoFile output/pca_related/protocol_example.related.for_pca.bed \
    --remove-samples output/pca_uf/protocol_example.pca_pheno.pca.projected.outliers \
    --name no_outlier

Re-run the QC on the **related** individuals after dropping any PCA-detected outliers (passed via `--remove-samples`), keeping the same variant set as the unrelated samples. The output PLINK bundle `protocol_example.related.for_pca.no_outlier.plink_qc.*` is the outlier-free, QC'd genotype set for the related individuals.

> Note: for this clean toy dataset no outliers are detected, so the removal list is empty and this step simply re-emits the QC'd related set.

Finally, merge back related and unrelated individuals as finalized genotype in PLINK format,

In [ ]:
sos run pipeline/genotype_formatting.ipynb merge_plink \
    --genoFile output/gwas_qc/genotype/protocol_example.merged.plink_qc.protocol_example.king.unrelated.plink_qc.prune.bed \
               output/pca_related/protocol_example.related.for_pca.no_outlier.plink_qc.bed \
    --cwd output/genotype_final \
    --name protocol_example.qced

You can stop here if you're analyzing a homogenous population.

## Optional — Analysis by population (admixed / multi-ancestry data)

The Steps 1–4 above produce a single PCA for the whole cohort, which is appropriate for a **homogeneous** population. If your cohort contains **multiple ancestry groups**, run the per-population workflow below instead: split the samples by population, then repeat the same QC → PCA → projection steps **separately within each population**.

The sub-steps below mirror Steps 1–4 exactly, just looped over each population (`for i in race1 race3`). Follow them in order:

| Sub-step | What it does |
| --- | --- |
| **P0. Split by population** | Write a per-population sample-ID list |
| **P1. QC unrelated** | Variant/sample QC within each population (unrelated samples) |
| **P2. Extract variants in related** | Keep the P1 pruned variants in the related samples |
| **P3. PCA on unrelated** | Run `flashpca` per population |
| **P4. Project related** | Project related samples onto each population’s PCA |
| **P5. Finalize** | Per-population QC to finalize genotypes |

In [ ]:
# Build per-population sample keep-lists from the phenotype table.
# One file per ancestry group, named output/pca_uf/protocol_example.ID.<race>.txt
# Each file holds two columns (FID, IID) with no header, matching plink's --keep format.
pheno <- read.table("input/covariate/protocol_example.pca_pheno.txt", header = TRUE, stringsAsFactors = FALSE)

for (i in 1:3) {
    race    <- subset(pheno, race == i)
    race_id <- cbind(race$FID, race$IID)
    write.table(race_id,
                paste0("output/pca_uf/protocol_example.ID.", i, ".txt"),
                quote = FALSE, sep = "\t", col.names = FALSE, row.names = FALSE)
}

### P1 — QC on unrelated samples, per population

Same as Step 2 (QC), but looped over each population. Below we only show Populations 1 and 3 as an example. **Next:** once each population has a QC’d unrelated set, extract those variants from the related samples (P2).

In [ ]:
for i in 3; do
    sos run pipeline/GWAS_QC.ipynb qc \
        --cwd output/pca_uf \
        --genoFile output/gwas_qc/genotype/protocol_example.merged.plink_qc.protocol_example.king.unrelated.plink_qc.bed \
        --keep-samples output/pca_uf/protocol_example.ID.$i.txt \
        --mac-filter 5 \
        --bad-ld True \
        --name pop_$i
done

### P2 — Extract the selected variants from related samples, per population

Same as the extract part of Step 4, looped per population (sample-level missingness filter only).

> Note: Population 1 may have **no related samples**, in which case its `*.related.for_pca_race1.filtered.extracted.stderr` log will say `Error: No people remaining after --keep.` That is expected — just continue with the populations that do have related samples. **Next:** run PCA on each population’s unrelated set (P3).

In [ ]:
for i in 3; do
    sos run pipeline/GWAS_QC.ipynb qc_no_prune \
        --cwd output/pca_uf \
        --genoFile output/pca_related/protocol_example.related.for_pca.bed \
        --keep-variants output/pca_uf/protocol_example.merged.plink_qc.protocol_example.king.unrelated.plink_qc.pop_$i.plink_qc.prune.in \
        --keep-samples output/pca_uf/protocol_example.ID.$i.txt \
        --maf-filter 0 --geno-filter 0 --mind-filter 0.1 --hwe-filter 0 \
        --name pop_$i
done

### P3 — Run PCA (flashpca) on unrelated samples, per population

Same as Step 3, looped per population. Here the number of PCs is set to 5 (`--k 5`) because each population’s sample size is small. **Next:** project the related samples onto each population’s PCA model (P4).

In [ ]:
for i in 3; do
    sos run pipeline/PCA.ipynb flashpca \
        --name pop_$i \
        --cwd output/pca_uf \
        --genoFile output/pca_uf/protocol_example.merged.plink_qc.protocol_example.king.unrelated.plink_qc.pop_$i.plink_qc.prune.bed \
        --phenoFile input/covariate/protocol_example.pca_pheno.txt \
        --label-col race \
        --pop-col race \
        --maha-k 2 \
        --k 5
done

### P4 — Project related samples onto each population’s PCA, per population

Same as Step 4 (projection), looped per population. Only populations that actually have related samples are projected (here, Population 3 only). **Next:** finalize genotypes per population (P5).

In [ ]:
for i in 3; do
    sos run pipeline/PCA.ipynb project_samples \
        --name pop_$i \
        --cwd output/pca_uf \
        --genoFile output/pca_uf/protocol_example.related.for_pca.pop_$i.plink_qc.extracted.bed \
        --phenoFile input/covariate/protocol_example.pca_pheno.txt \
        --pca-model output/pca_uf/protocol_example.pca_pheno.pop_$i.pca.rds \
        --label-col race \
        --pop-col race \
        --maha-k 2 \
        --k 5
done

To visualize the results at this point,

In [ ]:
%preview output/pca_uf/protocol_example.pca_pheno.pca.projected.pc.png

### P5 — Finalize genotypes per population

Same as the **Finalize genotype QC by PCA** step for a homogeneous population, applied per population and taking the detected outliers into account. See the `GWAS_QC.ipynb` documentation for the available QC options and recommendations. This is the end of the per-population workflow.

## Command Interface

In [ ]:
sos run PCA.ipynb -h

In [ ]:
[global]
# the output directory for generated files
parameter: cwd = path("output")
# A string to identify your analysis run
parameter: name = ""
# Name of the population column in the phenoFile
parameter: pop_col = ""
# Name of the populations (from the population column) you would like to plot and show on the PCA plot
parameter: pops = []
# Name of the color label column in the phenoFile; can be the same as population column. Can also be a separate column eg a "super population" column as a way to enable you to combine selected populations based on another column.
parameter: label_col = ""
# Number of Principal Components to output,must be consistant between flashpca run and project samples run (flashpca partial PCA method).
parameter: k = 20
# Number of Principal Components based on which outliers should be evaluated. Default is 5 but this should be based on examine the scree plot
parameter: maha_k = 5
# Homogeneity of populations. Set to --homogeneous when true and --no-homogeneous when false
parameter: homogeneous = False
# Software container option
parameter: container = ""
# For cluster jobs, number commands to run per job
parameter: job_size = 1
# Wall clock time expected
parameter: walltime = "5h"
# Memory expected
parameter: mem = "16G"
# Number of threads
parameter: numThreads = 10
suffix = '_'.join(pops)
cwd = path(f"{cwd:a}")
if not pop_col:
    homogeneous = True

# Determine if the file is in PLINK1 (BED/BIM/FAM) or PLINK2 (PGEN/PVAR/PSAM) format
def determine_plink_format(file_path):
    """
    Determine the PLINK file format based on file extensions and companion files.
    
    Args:
        file_path (str or Path): Path to the input file
    
    Returns:
        str: 'plink1' or 'plink2'
    """
    # Convert to string if it's a Path object
    file_path = str(file_path)
    
    # Check direct file extensions
    if file_path.endswith('.bed'):
        return 'plink1'
    elif file_path.endswith('.pgen'):
        return 'plink2'
    
    # If the file doesn't have a standard extension, try to infer format
    try:
        # Remove the file extension if present
        base_path = file_path.rsplit('.', 1)[0] if '.' in file_path else file_path
        
        # Check for PLINK1 companion files
        plink1_companion_files = [
            f"{base_path}.bim",
            f"{base_path}.fam"
        ]
        
        # Check for PLINK2 companion files
        plink2_companion_files = [
            f"{base_path}.pvar",
            f"{base_path}.psam"
        ]
        
        # Check PLINK1 format
        if all(os.path.exists(f) for f in plink1_companion_files):
            return 'plink1'
        
        # Check PLINK2 format
        if all(os.path.exists(f) for f in plink2_companion_files):
            return 'plink2'
    
    except Exception as e:
        print(f"Error determining PLINK format: {e}")
    
    # Default to PLINK1 if can't determine
    return 'plink1'

# Get the appropriate PLINK command prefix
def get_plink_command_prefix(file_path):
    format_type = determine_plink_format(file_path)
    if format_type == 'plink1':
        return "--bfile"
    else:  # plink2
        return "--pfile"


In [ ]:
# PCA command with PLINK, as a sanity check
[pca_plink]
# PLINK binary file
parameter: genoFile = path
input: genoFile
plink_command = get_plink_command_prefix(_input)
output: f'{cwd}/{genoFile:bn}.pca.eigenvec'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
bash: container = container, expand = "${ }", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout'
    plink2 ${plink_command} ${_input:n} --out ${_output:n} --pca ${k}

## PCA analysis

In [ ]:
# Run PCA analysis using flashpca 
[flashpca_1]
# Plink binary file
parameter: genoFile = path
# The phenotypic file
parameter: phenoFile = path(f'{genoFile}'.replace(".bed", ".fam").replace(".pgen", ".psam"))
# minimum population size to consider in the analysis
parameter: min_pop_size = 2
# How to standardize X before PCA
parameter: stand = "binom2"
## Input genoFile here is for unrelated samples
input: genoFile, phenoFile
output: f'{cwd}/{phenoFile:bn}{("."+name) if name else ""}.{(suffix+".") if suffix != "" else ""}pca.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: container = container, expand = "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout'
    # Load required libraries
    library(flashpcaR)
    library(dplyr)
    pops = c(${paths(pops):r,})
    f <- flashpca(${_input[0]:nr}, ndim=${k}, stand="${stand}", do_loadings=TRUE, check_geno=TRUE)
    rownames(f$loadings) <- read.table('${_input[0]:n}.bim',stringsAsFactors =F)[,2]
    # Use the projection file to generate pca plot
    pca <- as.data.frame(f$projection)
    # Preserve genotype sample IDs (FID:IID) as projection rownames so the
    # downstream merge by "ID" matches the phenotype table (toy MWE fix).
    .idfile <- if (file.exists(paste0("${_input[0]:n}", ".fam"))) paste0("${_input[0]:n}", ".fam") else paste0("${_input[0]:n}", ".psam")
    .ids <- read.table(.idfile, header=FALSE, comment.char="#", stringsAsFactors=FALSE)
    rownames(pca) <- paste(.ids[,1], .ids[,2], sep=":")
    pca <- tibble::rownames_to_column(pca, "ID")
    colnames(pca) <- c("ID",paste0("PC", 1:${k}))
  
    # Read fam file with phenotypes
    if(stringr::str_detect(${_input[1]:r},"\\.fam$|\\.psam$")){
        pheno <- read.table(${_input[1]:r}, header=F,stringsAsFactors =F)
        colnames(pheno) = c("FID", "IID", "MID", "PID", "SEX", "STATUS")
    } else {
        pheno <- read.table(${_input[1]:r}, header=T,stringsAsFactors =F)
        if(!"IID" %in% colnames(pheno)) stop("No IID column in the phenoFile. Please rename the header of the phenoFile")
        if(!"FID" %in% colnames(pheno)) pheno$FID = pheno$IID
    }
    # Make the unique ID by merge FID and IID
    pheno$ID = paste(pheno$FID,pheno$IID,sep = ":")
    #check duplicated ID
    if(length(unique(pheno$ID))!=length(pheno$ID)) stop("There are duplicated names in IID column of phenoFile")

    if (length(pops)>0) pheno <- pheno %>%filter(${pop_col if pop_col else  "pop"} %in% pops | ${label_col if label_col else  "pop"} %in% pops)
    pca <-merge(pheno, pca,by ="ID", all=FALSE) 
    #
    if (${"TRUE" if pop_col else "FALSE"}) {
        # remove populations have less than ${min_pop_size} samples
        pop<-names(table(pca$${pop_col if pop_col else "pop"}))
        pop_filter<-pop[table(pca$pop)<${min_pop_size}] # pop to be removed
        if (length(pop_filter)>0) {
            warning(for (i in pop_filter){cat(i,';')},'these ', length(pop_filter)," population will be removed due to having less than ${min_pop_size} samples in data.")
            # remove
            pca<-pca%>% filter(${f'!{pop_col}%in%pop_filter' if pop_col else pop_col})
        }
    } else {
      pca$pop <- 1 
    }

    # Write the PC scores to a file
    write.table(pca,"${_output:n}.txt", sep="\t", quote=FALSE, row.names=FALSE, col.names=TRUE)
    dat = list(pca_model = f, pc_scores = pca, meta = "${_input[1]:bn} ${suffix}")
    # compute centroids before projecting back the samples
    # (calculate mean/median/cov per pop)
    if(${"FALSE" if homogeneous else "TRUE"}){
        pop_group <- split(dat$pc_scores[ ,c(paste0("PC", 1:${maha_k}))], list(Group = dat$pc_scores$${pop_col if pop_col else "pop"}))
        dat$pc_cov <- lapply(pop_group, function(x) cov(x))
        dat$pc_mean <- lapply(pop_group, function(x) sapply(x, mean))
        dat$pc_median <- lapply(pop_group, function(x) sapply(x, median))
    } else {
        dat$pc_cov <- cov(f$projection[,1:${maha_k}])
        dat$pc_mean <- apply(f$projection[,1:${maha_k}], 2, mean)
        dat$pc_median <- apply(f$projection[,1:${maha_k}], 2, median)
    }
  
    # save results
    saveRDS(dat, ${_output:r})
  
bash: expand= "$[ ]", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container
        stdout=$[_output:n].stdout
        for i in $[_output] ; do 
        echo "output_info: $i " >> $stdout;
        echo "This rds file is a list containing the pca for unrelated sample" >> $stdout;
        echo "output_size:" `ls -lh $i | cut -f 5  -d  " "`   >> $stdout;
        done

## Project related individuals back

In [ ]:
# Project back to PCA model additional samples
[project_samples_1]
# Plink binary file
parameter: genoFile = path
# The phenotypic file
parameter: phenoFile = path(f'{genoFile}'.replace(".bed", ".fam").replace(".pgen", ".psam"))
parameter: pca_model = f'{cwd}/{phenoFile:bn}{("."+name) if name else ""}.{(suffix+".") if suffix != "" else ""}pca.rds'
## Input genoFile here is for related samples
input: genoFile, phenoFile, pca_model
output: f'{cwd}/{phenoFile:bn}{("."+name) if name else ""}.pca.projected.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: container=container, expand= "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout'
    # Load required libraries
    library(dplyr)
    library(flashpcaR)
    # Read the PLINK binary files
    frel <- ${_input[0]:nr}
    # Read loadings, center and scale from previous PCA
    dat <- readRDS(${_input[2]:r})
    f <- dat$pca_model                           
    # Read the bim/pvar file to obtain reference alleles
    variant_file <- if(stringr::str_detect('${_input[0]}', "\\.bed$")) '${_input[0]:n}.bim' else '${_input[0]:n}.pvar'
    bim <- read.table(variant_file, stringsAsFactors=F)
    ref_col <- if(stringr::str_detect(variant_file, "\\.bim$")) 5 else 4  # Column index differs between .bim and .pvar
    ref <- as.character(bim[,ref_col])
    names(ref) <- bim[,2]
    if (nrow(f$loadings) != length(ref)) stop("Input number of variants should be the same as used in the previous PCA model")
    overlapped_variants <- match(names(ref), rownames(f$loadings))
    # Project related samples
    fpro <- project(frel, loadings=f$loadings[overlapped_variants,], orig_mean=f$center[overlapped_variants], orig_sd=f$scale[overlapped_variants], ref_allele=ref)
    pca <- fpro$projection
    k = ${k}
    pca <- as.data.frame(fpro$projection)
    # Preserve genotype sample IDs (FID:IID) as projection rownames so the
    # downstream merge by "ID" matches the phenotype table (toy MWE fix).
    .idfile <- if (file.exists(paste0("${_input[0]:n}", ".fam"))) paste0("${_input[0]:n}", ".fam") else paste0("${_input[0]:n}", ".psam")
    .ids <- read.table(.idfile, header=FALSE, comment.char="#", stringsAsFactors=FALSE)
    rownames(pca) <- paste(.ids[,1], .ids[,2], sep=":")
    pca <- tibble::rownames_to_column(pca, "ID")
    colnames(pca) <- c("ID",paste0("PC", 1:k))
    
    # Read fam/psam file with phenotypes
    if(stringr::str_detect(${_input[1]:r},"\\.fam$|\\.psam$")){
        pheno <- read.table(${_input[1]:r}, header=F,stringsAsFactors =F)
        colnames(pheno) = c("FID", "IID", "MID", "PID", "SEX", "STATUS")
    } else {
        pheno <- read.table(${_input[1]:r}, header=T,stringsAsFactors =F)
        if(!"IID" %in% colnames(pheno)) stop("No IID column in the phenoFile. Please rename the header of the phenoFile")
        if(!"FID" %in% colnames(pheno)) pheno$FID = pheno$IID
    }
    # Make the unique ID by merge FID and IID
    pheno$ID = paste(pheno$FID,pheno$IID,sep = ":")
    #check duplicated ID
    if(length(unique(pheno$ID))!=length(pheno$ID)) stop("There are duplicated names in IID column of phenoFile")

    pca <-merge(pheno, pca,by ="ID", all=FALSE)
    dat$pc_scores = bind_rows(dat$pc_scores, pca)
    
    # Write the PC scores to a file
    write.table(dat$pc_scores,"${_output:n}.txt", sep="\t", quote=FALSE, row.names=FALSE, col.names=TRUE)
    # save results
    saveRDS(dat, ${_output:r})
  
bash: expand= "$[ ]", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container
        stdout=$[_output:n].stdout
        for i in $[_output] ; do 
        echo "output_info: $i " >> $stdout;
        echo "This rds file is a list containing the pca for projected sample" >> $stdout;
        echo "output_size:" `ls -lh $i | cut -f 5  -d  " "`   >> $stdout;
        done
        for i in $[_output:n].txt ; do 
        echo "output_info $i " >> $stdout;
        echo "This is the PC score" >> $stdout;
        echo "output_size:" `ls -lh $i | cut -f 5  -d  " "`   >> $stdout;
        echo "output_rows:" `cat $i | wc -l  | cut -f 1 -d " "`   >> $stdout;
        echo "output_column:" `cat $i | head -1 | wc -w `   >> $stdout;
        echo "output_preview:"   >> $stdout;
        cat $i | head  | cut -f 1,2,3,4,5,6   >> $stdout ; done

## Plot PCA results


In [ ]:
# Plot PCA results. 
# Can be used independently as "plot_pca" or combined with other workflow as eg "flashpca+plot_pca"
[plot_pca]
parameter: outlier_file = path()
parameter: plot_data = path
parameter: min_axis = ""
parameter: max_axis = ""
input: plot_data
output: f'{cwd}/{_input:bn}.pc.png',
        f'{cwd}/{_input:bn}.scree.png',
        f'{cwd}/{_input:bn}.scree.txt'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = 1, tags = f'{step_name}_{_output[0]:bn}'
R: container = container, volumes = [f"{outlier_file:ad}:{outlier_file:ad}"], expand = "${ }", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout'
    library(dplyr)
    library(ggplot2)
    library(gridExtra)
    library(matrixStats)
    pops = c(${paths(pops):r,})
    dat = readRDS(${_input:r})
    f = dat$pca_model
    pca_final<-dat$pc_scores
    
    if (length(pops)>1) pca_final <- pca_final %>% filter(${pop_col if pop_col else "pop"} %in% pops | ${label_col if label_col else  "pop"} %in% pops) 
    pca_final <- pca_final %>% mutate(${label_col if label_col else "pop"}=as.character(${label_col if label_col else  "pop"}))
    k = ${k}
    # manually set colors for PCA plotting, to avoid similar colors in one plot:
    # generated by https://mokole.com/palette.html
    set.seed(999)
    colors_40 = sample(c("#a9a9a9", "#2f4f4f", "#556b2f", "#a0522d", "#7f0000", "#006400", "#808000", "#483d8b", "#3cb371", "#bdb76b", "#4682b4", "#9acd32", 
                          "#20b2aa", "#00008b", "#32cd32", "#daa520", "#7f007f", "#b03060", "#ff0000", "#ff8c00", "#ffff00", "#0000cd", "#00ff00", "#9400d3", 
                          "#00fa9a", "#00ffff", "#00bfff", "#f4a460", "#f08080", "#adff2f", "#ff6347", "#ff00ff", "#1e90ff", "#dda0dd", "#7b68ee", "#afeeee", 
                          "#ee82ee", "#ff69b4", "#ffe4c4", "#ffc0cb"))
    colors_20 = sample(c("#2f4f4f", "#2e8b57", "#8b0000", "#808000", "#00008b", "#ff0000", "#ff8c00", "#00ff00", "#4169e1", "#00ffff", "#00bfff", "#0000ff", 
                          "#da70d6", "#d8bfd8", "#ff00ff", "#eee8aa", "#ffff54", "#ff1493", "#ffa07a", "#98fb98"))

    # assign colors to each ethnicity:
    num_col=length(unique(pca_final$${label_col if label_col else "pop"}))
    if (num_col <= 20) {
       color_list <- colors_20[1:num_col]
    } else {
       color_list <- colors_40[1:num_col]
    }

    ###
    # Make the plots
    ###
    # Get the min and max values for x and y-axes
    if (${"TRUE" if len(min_axis) == 0 or len(max_axis) == 0 else "FALSE"}) {
        min_axis <- round(colMins(as.matrix(f$projection[sapply(f$projection, is.numeric)])),1)
        max_axis <- round(colMaxs(as.matrix(f$projection[sapply(f$projection, is.numeric)])),1)  
    } else {
        min_axis <- as.double(${min_axis})
        max_axis <- as.double(${max_axis})
    }
    if (${"TRUE" if outlier_file.is_file() else "FALSE"}) {
        outliers <- read.table(${outlier_file:r}, col.names=c("FID", "IID"),stringsAsFactors =F)
        plot_pcs = function(pca_final, x, y, title="") {
            ggplot(pca_final, aes_string(x=x, y=y)) + geom_point(${f'aes(color={label_col})' if label_col else ""}) + 
              # add circles for these ouliters:
              geom_point(data=filter(pca_final, IID %in% outliers$IID, FID %in% outliers$FID), shape = 21, size=1.5, color='red', stroke = 0.9) +
              # add outliers dots:
              geom_point(data=filter(pca_final, IID %in% outliers$IID, FID %in% outliers$FID), shape = 16, size=1${f',aes(color={label_col})' if label_col else ""} ) + 
              labs(title=title,x=x, y=y) +
              scale_y_continuous(limits=c(min_axis, max_axis)) +
              scale_x_continuous(limits=c(min_axis, max_axis)) +
              scale_color_manual(values=color_list) +
              theme_classic()
        }} else {
        plot_pcs = function(pca_final, x, y, title="") {
          ggplot(pca_final, aes_string(x=x, y=y)) + geom_point(${f'aes(color={label_col})' if label_col else ""}) + 
              labs(title=title,x=x, y=y) +
              scale_y_continuous(limits=c(min_axis, max_axis)) +
              scale_x_continuous(limits=c(min_axis, max_axis)) +
              scale_color_manual(values=color_list) +
              theme_classic()
        }}
    unit = 4
    n_col = min(4, k)
    n_row = ceiling(k / n_col)
    plots = lapply(1:(k-1), function(i) plot_pcs(pca_final, paste0("PC",i), paste0("PC",i+1), dat$meta))
    png('${_output[0]}', width = unit * n_col, height = unit * n_row, unit='in', res=300)
    do.call(gridExtra::grid.arrange, c(plots, list(ncol = n_col, nrow = n_row)))
    dev.off()
    # Create scree plot
    PVE <- f$values
    PVE <- round(PVE/sum(PVE), 2)
    PVEplot <- qplot(c(1:length(PVE)), PVE) + geom_line() + xlab("Principal Component") + ylab("PVE") + ggtitle("Scree Plot") + ylim(0, 1) +scale_x_discrete(limits=factor(1:length(PVE)))
    PVE_cum <- cumsum(PVE)/sum(PVE)
    cumPVEplot <- qplot(c(1:length(PVE)), cumsum(PVE)) + geom_line() + xlab("Principal Component") + ylab("PVE") + ggtitle("Cumulative PVE Plot") + ylim(0, 1) + scale_x_discrete(limits=factor(1:length(PVE)))
    png('${_output[1]}', width = 8, height = 4, unit='in', res=300)
    grid.arrange(PVEplot, cumPVEplot, nrow = 1)
    dev.off()
    ## Textual Output
    
    PVE_output = tibble(PCs = 1:length(PVE), PVE = PVE, PVE_cum = PVE_cum )
    PVE_output%>%readr::write_delim(${_output[2]:r},"\t")

bash: expand= "$[ ]", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout' , container = container
        stdout=$[_output[0]:n].stdout
        for i in $[_output[2]] ; do 
        echo "output_info $i " >> $stdout;
        echo "This is the PC score" >> $stdout;
        echo "output_size:" `ls -lh $i | cut -f 5  -d  " "`   >> $stdout;
        echo "output_rows:" `cat $i | wc -l  | cut -f 1 -d " "`   >> $stdout;
        echo "output_column:" `cat $i | head -1 | wc -w `   >> $stdout;
        echo "output_preview:"   >> $stdout;
        cat $i | head  | cut -f 1,2,3,4,5,6   >> $stdout ; done

## Detect outliers

In [ ]:
# Calculate Mahalanobis distance per population and report outliers
[detect_outliers]
# Set the probability to remove outliers eg 0.95 or 0.997
parameter: prob = 0.997
# Mahalanobis distance p-value cutoff
parameter: pval = 0.05
# Robust Mahalanobis to outliers
parameter: robust = True
parameter: pca_result = path
input: pca_result
output: distance=f'{_input:n}.mahalanobis.rds',
        identified_outliers=f'{_input:n}.outliers',
        analysis_summary=f'{_input:n}.analysis_summary.md',
        qqplot_mahalanobis=f'{_input:n}.mahalanobis_qq.png',
        hist_mahalanobis=f'{_input:n}.mahalanobis_hist.png'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = 1, tags = f'{step_name}_{_output[0]:bn}'
bash: container = container, expand = "${ }"
    echo '''---
    theme: base-theme
    style: |
      img {
        height: 80%;
        display: block;
        margin-left: auto;
        margin-right: auto;
      }
    ---    
    ''' > ${_output[2]}
    
R: container = container, expand= "${ }", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout'
    # Load required libraries
    library(dplyr)
    library(ggplot2)
    library(gridExtra)
  
    # invert a known covariance matrix but allow them to be numerically singular matrix (still assuming full rank)
    robust_inv = function(s) {
        #tryCatch(solve(s), error=function(cond) solve(Matrix::nearPD(s)$mat))
        tryCatch(solve(s), error=function(cond) MASS::ginv(s))
    }

    # Calculate mahalanobis distance  
    calc_mahalanobis_dist = function(x, m, s, name = '', prob=${prob}) {
        pc <- x %>%
          select("IID","FID", "${pop_col if pop_col else "pop"}", starts_with("PC"))
        mu_pc <- pc[,4:(4 + length(m) - 1)]
        pc$mahal = mahalanobis(mu_pc, m, robust_inv(s), inverted=TRUE)
        pc$p <- pchisq(pc$mahal, df=nrow(s), lower.tail=FALSE)
        manh_dis_sq_cutoff = quantile(pc$mahal, probs=prob)
        # Obtain outliers
        outliers = pc[(pc$mahal > manh_dis_sq_cutoff & pc$p < ${pval}),]
        d_summary = paste0(capture.output(summary(pc$mahal)), collapse = '\n')
        msg = paste('#', name, "result summary\n## Mahalanobis distance summary:\n```\n", d_summary, "\n```\n", 
            paste("The cut-off for outlier removal is set to:", manh_dis_sq_cutoff, "and the number of individuals to remove is:", nrow(outliers),"\n"),
            paste("The new sample size after outlier removal is:",nrow(pc) - nrow(outliers),"\n"))
        #
        outliers <- outliers %>%
        select(FID,IID)
      list(pc=pc, manh_dis_sq_cutoff=manh_dis_sq_cutoff, msg=msg, outliers=outliers)
    }

    dat = readRDS(${_input:r})
    if (is.list(dat$pc_mean)) {
      pops = names(dat$pc_mean)
      pop_group = split(dat$pc_scores, f = dat$pc_scores$${pop_col if pop_col else "pop"})
      res = lapply(pops, function(p) calc_mahalanobis_dist(pop_group[[p]], dat$${"pc_mean" if not robust else "pc_median"}[[p]], dat$pc_cov[[p]], name = paste(dat$meta, p)))
      names(res) = pops
      res = list(
          msg = do.call(paste, c(lapply(pops, function(p) res[[p]]$msg), sep = "\n")),
          manh_dis_sq_cutoff = cbind(pops, sapply(pops, function(p) res[[p]]$manh_dis_sq_cutoff)),
          outliers = do.call(rbind, c(lapply(pops, function(p) res[[p]]$outliers))),
          pc = do.call(rbind, c(lapply(pops, function(p) res[[p]]$pc)))
          )
    } else {
      res = calc_mahalanobis_dist(dat$pc_scores, dat$${"pc_mean" if not robust else "pc_median"}, dat$pc_cov, name = dat$meta)
    }
      
    write(res$msg, ${_output[2]:r})   
    # Plot mahalanobis
    k = ${k}
    png('${_output[3]}', width = 4, height = 4, unit='in', res=300)
    qqplot(qchisq(ppoints(100), df=k), res$pc$mahal, main = expression("Mahalanobis" * ~D^2 * " vs. quantiles of" * ~ chi[k]^2), xlab = expression(chi[2]^2 * ", probability points = 100"), ylab = expression(D^2), pch=16)
    abline(0,1,col='red')
    dev.off() 
    png('${_output[4]}', width = 4, height = 4, unit='in', res=300)
    ggplot(res$pc, aes(x=mahal)) + geom_histogram(aes(y = ..count..), binwidth = 0.5, colour = "#1F3552", fill = "#4271AE") + scale_x_continuous(name = "Mahalanobis distance") + theme_classic()
    dev.off()
  
    # Save results and outliers
    saveRDS(res,${_output[0]:r})
    write.table(res$outliers, ${_output[1]:r}, sep="\t", quote=FALSE, row.names=FALSE, col.names=FALSE)
  
bash: expand= "$[ ]", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout', container = container
        stdout=$[_output[0]:n].stdout
        for i in $[_output[0]] ; do 
        echo "output_info: $i " >> $stdout;
        echo "This rds file detail analysis output for detect outliers" >> $stdout;
        echo "output_size:" `ls -lh $i | cut -f 5  -d  " "`   >> $stdout;
        done
        for i in $[_output[1]] ; do 
        echo "output_info: $i " >> $stdout;
        echo "This text file documents the outliers samples" >> $stdout;
        echo "output_size:" `ls -lh $i | cut -f 5  -d  " "`   >> $stdout;
        done
        for i in $[_output[2]] ; do 
        echo "output_info: $i " >> $stdout;
        echo "This text file documents the analysis summary" >> $stdout;
        echo "output_size:" `ls -lh $i | cut -f 5  -d  " "`   >> $stdout;
        done
        for i in $[_output[3]] ; do 
        echo "output_info: $i " >> $stdout;
        echo "This plot documents the mahalanobis analysis result" >> $stdout;
        echo "output_size:" `ls -lh $i | cut -f 5  -d  " "`   >> $stdout; 
        done
        for i in $[_output[4]] ; do 
        echo "output_info: $i " >> $stdout;
        echo "This plot documents the mahalanobis distribution result" >> $stdout;
        echo "output_size:" `ls -lh $i | cut -f 5  -d  " "`   >> $stdout;
        done

## Add plot and outlier detection to PCA steps

## Anticipated Results

The pipeline produces output files in the `output/` subdirectory named after the workflow step. Verify success by checking that output files exist and are non-empty. See the **Output** section above for the expected file names and formats.

In [ ]:
[flashpca_2, project_samples_2]
# Set the probability to remove outliers eg 0.95 or 0.997
parameter: prob = 0.997
# Robust Mahalanobis to outliers
parameter: robust = True
output: distance=f'{_input:n}.mahalanobis.rds',
        identified_outliers=f'{_input:n}.outliers',
        analysis_summary=f'{_input:n}.analysis_summary.md',
        qqplot_mahalanobis=f'{_input:n}.mahalanobis_qq.png',
        hist_mahalanobis=f'{_input:n}.mahalanobis_hist.png'
sos_run("detect_outliers", pca_result=_input, prob=prob, robust=robust)

[flashpca_3, project_samples_3]
input: output_from(1), output_from(2)['identified_outliers']
outliers = [x.strip() for x in open(_input[1]).readlines() if x.strip()]
output: f"{cwd}/{_input[0]:bn}.pc.png",
        f"{cwd}/{_input[0]:bn}.scree.png"
sos_run("plot_pca", plot_data = _input[0], outlier_file = _input[1] if len(outliers) else path())